# PyTorch MLP with epoch-level learning curves

This is a separate reimplementation of the selected three-feature MLP,
created to record both loss and accuracy at every epoch. It does not
replace the legacy `sklearn` model used in the existing analysis.

The data split and input columns are unchanged: `mean_o`, `std_o`, and
`skew_o`. The network is `3 -> 64 -> 32 -> 1` with `ReLU` activations.
The epoch history uses the fixed threshold `0.5`; after training, the
final operating threshold is selected by maximum validation F1 and then
applied once to the test set.


In [ ]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


WORK_ROOT = Path.cwd().resolve()

DATA_FOLDER = Path("/hercules/results/akazantsev/rfim_dataset")
META_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels_meta.csv"
SPLIT_PATH = DATA_FOLDER / "split_indices.npz"
PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels.npy"

# The training subset deliberately retains only statistical features and labels.
# These full files retain channel and segment identity and are used only by the
# inference-timing notebook, where one input must correspond to a real 256-channel
# observation segment.
FULL_META_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels_meta.csv"
FULL_PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels.npy"
SUBSET_SOURCE_INDICES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_indices.npy"

# Change the tag only for a deliberate new experiment. Existing results are never overwritten.
RUN_TAG = "b0531_legacy_performance_v1"
RUN_ROOT = WORK_ROOT / "outputs" / "performance_comparison" / RUN_TAG


def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def write_json(path: Path, payload: dict) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(json_ready(payload), handle, indent=2, sort_keys=True)
        handle.write("\n")


def git_revision() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=WORK_ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    log_loss,
    precision_recall_fscore_support,
    roc_auc_score,
)

RANDOM_STATE = 42
SELECTED_FEATURES = ["mean_o", "std_o", "skew_o"]
EPOCHS = 200
BATCH_SIZE = 200  # sklearn MLPClassifier uses min(200, n_samples) by default.
LEARNING_RATE = 1e-3
LEGACY_ALPHA = 1e-4
# Keep the default on CPU: the legacy sklearn MLP was trained on CPU.
# A CUDA run is possible, but should use a distinct RUN_TAG and be
# reported as a separate hardware configuration.
RUN_DEVICE = "cpu"
if RUN_DEVICE not in {"cpu", "cuda"}:
    raise ValueError("RUN_DEVICE must be either 'cpu' or 'cuda'.")
if RUN_DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("RUN_DEVICE='cuda' was requested, but CUDA is not available.")
DEVICE = torch.device(RUN_DEVICE)
output_dir = RUN_ROOT / "mlp_pytorch_orig_top3"

if output_dir.exists():
    raise FileExistsError(
        f"{output_dir} already exists. Choose a new RUN_TAG rather than overwrite it."
    )
output_dir.mkdir(parents=True)

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(RANDOM_STATE)
    print("GPU:", torch.cuda.get_device_name(DEVICE))
print("Device:", DEVICE)


def best_threshold_by_f1(y_true: np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    best_threshold, best_f1 = 0.5, -1.0
    for threshold in np.linspace(0.01, 0.99, 99):
        prediction = (scores >= threshold).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(
            y_true, prediction, average="binary", zero_division=0
        )
        if f1 > best_f1:
            best_threshold, best_f1 = float(threshold), float(f1)
    return best_threshold, best_f1


def eval_binary(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    prediction = (scores >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, prediction, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    both_classes = len(np.unique(y_true)) == 2
    return {
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, prediction)),
        "precision": float(precision), "recall": float(recall), "f1": float(f1),
        "roc_auc": float(roc_auc_score(y_true, scores)) if both_classes else None,
        "pr_auc": float(average_precision_score(y_true, scores)) if both_classes else None,
        "logloss": float(log_loss(y_true, np.c_[1 - scores, scores], labels=[0, 1])) if both_classes else None,
    }


In [ ]:
# Keep the original subset paths and split keys. Missing values are
# replaced by zero, exactly as in the legacy SimpleImputer pipeline.
meta = pd.read_csv(META_PATH)
meta["label"] = meta["label"].fillna("None")
splits = np.load(SPLIT_PATH)
train_idx = np.asarray(splits["train_idx"], dtype=int)
val_idx = np.asarray(splits["val_idx"], dtype=int)
test_idx = np.asarray(splits["test_idx"], dtype=int)

missing_features = set(SELECTED_FEATURES).difference(meta.columns)
if missing_features:
    raise ValueError(f"Metadata is missing selected features: {sorted(missing_features)}")

x_all = (
    meta[SELECTED_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .to_numpy(dtype=np.float32)
)
y_all = meta["label"].eq("NBRFI").to_numpy(dtype=np.float32)

x_train, x_validation, x_test = (x_all[indices] for indices in (train_idx, val_idx, test_idx))
y_train, y_validation, y_test = (y_all[indices] for indices in (train_idx, val_idx, test_idx))

datasets = {
    "train": TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    "validation": TensorDataset(torch.from_numpy(x_validation), torch.from_numpy(y_validation)),
    "test": TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
}
loaders = {
    "train": DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True),
    "validation": DataLoader(datasets["validation"], batch_size=BATCH_SIZE, shuffle=False),
    "test": DataLoader(datasets["test"], batch_size=BATCH_SIZE, shuffle=False),
}

print("Selected features:", SELECTED_FEATURES)
print("Train / validation / test:", len(train_idx), len(val_idx), len(test_idx))


In [ ]:
class MLPOrigTop3Logits(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, features):
        return self.layers(features).squeeze(-1)


model = MLPOrigTop3Logits().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()

# sklearn alpha is an L2 term in the average-loss objective. This
# scaling is a close PyTorch analogue, but not an exact reproduction of
# sklearn's optimiser or parameter initialization.
weight_decay = LEGACY_ALPHA / len(train_idx)
optimizer = torch.optim.Adam(
    model.parameters(), lr=LEARNING_RATE, weight_decay=weight_decay
)


def run_epoch(loader, training: bool) -> dict:
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_rows = 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for features, labels in loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(features)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            total_correct += ((torch.sigmoid(logits) >= 0.5) == labels.bool()).sum().item()
            total_rows += len(labels)
    return {"loss": total_loss / total_rows, "accuracy": total_correct / total_rows}


@torch.no_grad()
def predict_scores(loader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    scores, labels = [], []
    for features, target in loader:
        logits = model(features.to(DEVICE))
        scores.append(torch.sigmoid(logits).cpu().numpy())
        labels.append(target.numpy())
    return np.concatenate(labels).astype(int), np.concatenate(scores)


In [ ]:
history = []
best_epoch = None
best_validation_accuracy = -np.inf
checkpoint_path = output_dir / "best_checkpoint.pt"
optimizer_wall_clock_s = 0.0

full_loop_started = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    optimizer_started = time.perf_counter()
    train = run_epoch(loaders["train"], training=True)
    optimizer_wall_clock_s += time.perf_counter() - optimizer_started
    validation = run_epoch(loaders["validation"], training=False)
    history.append({
        "epoch": epoch,
        "train_loss": train["loss"],
        "validation_loss": validation["loss"],
        "train_accuracy": train["accuracy"],
        "validation_accuracy": validation["accuracy"],
    })

    if validation["accuracy"] > best_validation_accuracy:
        best_epoch = epoch
        best_validation_accuracy = validation["accuracy"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "validation_accuracy": validation["accuracy"],
            "feature_cols": SELECTED_FEATURES,
        }, checkpoint_path)

    if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
        print(
            f"Epoch {epoch:3d}/{EPOCHS}: "
            f"train loss={train['loss']:.5f}, validation loss={validation['loss']:.5f}, "
            f"validation accuracy={validation['accuracy']:.4f}"
        )

train_and_validation_wall_clock_s = time.perf_counter() - full_loop_started

checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
if checkpoint["feature_cols"] != SELECTED_FEATURES:
    raise ValueError("Checkpoint feature order does not match the declared feature order.")
model.load_state_dict(checkpoint["model_state_dict"])

y_validation_int, validation_scores = predict_scores(loaders["validation"])
threshold, validation_f1 = best_threshold_by_f1(y_validation_int, validation_scores)
y_test_int, test_scores = predict_scores(loaders["test"])
test_metrics = eval_binary(y_test_int, test_scores, threshold)

history_path = output_dir / "history.csv"
pd.DataFrame(history).to_csv(history_path, index=False)
pd.DataFrame({
    "subset_row_index": test_idx,
    "label": y_test_int,
    "score": test_scores,
    "prediction_at_validation_threshold": (test_scores >= threshold).astype(int),
}).to_csv(output_dir / "test_predictions.csv", index=False)

print(f"Optimizer-only time: {optimizer_wall_clock_s:.3f} s")
print(f"Train-and-validation loop time: {train_and_validation_wall_clock_s:.3f} s")
print(f"Best epoch: {best_epoch}; validation accuracy: {best_validation_accuracy:.4f}")
print(f"Validation-selected threshold: {threshold:.2f}; validation F1: {validation_f1:.4f}")
print(f"Test F1: {test_metrics['f1']:.4f}")


In [ ]:
summary = {
    "run_tag": RUN_TAG,
    "model": "MLPOrigTop3Logits",
    "architecture": "Linear(3,64) -> ReLU -> Linear(64,32) -> ReLU -> Linear(32,1)",
    "selected_features": SELECTED_FEATURES,
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None,
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "code_revision": git_revision(),
    "dataset": {
        "metadata_path": META_PATH,
        "split_path": SPLIT_PATH,
        "n_train": len(train_idx),
        "n_validation": len(val_idx),
        "n_test": len(test_idx),
    },
    "training_protocol": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "loss": "BCEWithLogitsLoss",
        "legacy_alpha": LEGACY_ALPHA,
        "weight_decay": weight_decay,
        "checkpoint_selection": "maximum_validation_accuracy",
        "epoch_accuracy_threshold": 0.5,
        "final_threshold_selection": "maximum_validation_f1 over [0.01, 0.99] in steps of 0.01",
    },
    "optimizer_wall_clock_s": optimizer_wall_clock_s,
    "train_and_validation_wall_clock_s": train_and_validation_wall_clock_s,
    "best_epoch": best_epoch,
    "best_validation_accuracy": best_validation_accuracy,
    "threshold": threshold,
    "validation_f1": validation_f1,
    "test_metrics": test_metrics,
    "artifacts": {
        "checkpoint": checkpoint_path,
        "history": history_path,
        "test_predictions": output_dir / "test_predictions.csv",
    },
}
write_json(output_dir / "training_summary.json", summary)
print(json.dumps(json_ready(summary), indent=2))
